Для разметки будем исползовать - Qwen2.5-VL-7B-Instruct

### Стратегия Разметки

Мы создадим **три типа вопросов**:

1.  **Text-Only QA (Теория):**
    *   *Вход:* Чанк текста из `theory.json`.
    *   *Промпт:* "Сформулируй технический вопрос, ответ на который содержится в этом тексте. И дай ответ."
2.  **Visual-Only QA (Топология):**
    *   *Вход:* Картинка схемы из `images.json`.
    *   *Промпт:* "Посмотри на схему. Какой элемент подключен к...? / Какого типа источник?"
3.  **Multimodal QA (Reasoning):**
    *   *Вход:* Картинка + Текст, который был перед ней.
    *   *Промпт:* "Используя схему и описание, объясни принцип работы..."




In [1]:
from google.colab import drive
import os
import torch

drive.mount('/content/drive')


BASE_DIR = "/content/drive/MyDrive/EE_RAG"
DATA_DIR = os.path.join(BASE_DIR, "data")

!pip install --upgrade pip -q
# 1. свежая версия для Qwen2.5
!pip install -q git+https://github.com/huggingface/transformers

!pip install -q -U transformers accelerate qwen-vl-utils
!pip uninstall -y sentence-transformers
print(f"Данные: {DATA_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Данные: /content/drive/MyDrive/EE_RAG/data


Загрузка модели (Qwen2.5-VL-7B-Instruct)

In [2]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Используем bfloat16 для A100 (вроде быстрее и точнее)
# flash_attention_2 - критически важен для длинных контекстов
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"


model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_name)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section'}


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Пишем промпты для разных типов данных. Модель будет возвращать json

In [3]:
import json
import random
from PIL import Image

def generate_qa_pair(item, mode="text", preceding_text=None):
    """
    mode: 'text', 'image', 'hybrid'
    """

    if mode == "text":
        user_content = [
            {"type": "text", "text": f"Текст учебника:\n{item['text']}\n\nЗАДАЧА: Придумай 1 конкретный вопрос по электротехнике, на который можно ответить, опираясь ТОЛЬКО на этот текст. Не используй общие фразы. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]
    elif mode == "image":
        user_content = [
            {"type": "image", "image": item['path']},
            {"type": "text", "text": "Это электрическая схема. Внимательно проанализируй её топологию (связи элементов). Придумай 1 вопрос о том, как соединены элементы или что изображено. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]
    elif mode == "hybrid":
        # Склеиваем описание и картинку
        context_text = preceding_text if preceding_text else "Описание отсутствует."
        user_content = [
            {"type": "image", "image": item['path']},
            {"type": "text", "text": f"Текст описания:\n{context_text}\n\nЗАДАЧА: Используя И схему, И текст, придумай сложный вопрос на понимание. Верни ответ в формате JSON: {{'question': '...', 'answer': '...'}}"}
        ]

    messages = [{"role": "user", "content": user_content}]

    # 3. Инференс
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    # Генерация
    generated_ids = model.generate(**inputs, max_new_tokens=256)

    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]


    if "assistant\n" in output_text:
        output_text = output_text.split("assistant\n")[-1]

    # Попытка парсинга JSON
    try:
        # Убираем markdown обертку ```json ... ```
        clean_json = output_text.replace("```json", "").replace("```", "").strip()
        qa = json.loads(clean_json)
        return qa
    except:
        return None


Тестовый прогон на 10 семплах:

In [4]:
import random
import json

with open(os.path.join(DATA_DIR, "theory.json"), 'r') as f:
    theory_data = json.load(f)
with open(os.path.join(DATA_DIR, "images.json"), 'r') as f:
    images_data = json.load(f)

text_lookup = {t['id']: t['text'] for t in theory_data}

samples_text = random.sample(theory_data, 5)
valid_images = [img for img in images_data if os.path.exists(img['path'])]
samples_img = random.sample(valid_images, 5)

test_dataset = []


print("Generating Text Questions")
for item in samples_text:
    qa = generate_qa_pair(item, mode="text")
    if qa:
        entry = {
            "question": qa['question'],
            "answer": qa['answer'],
            "type": "text",
            "gold_ids": [item['id']], # Правильный ответ - этот чанк текста
            "metadata": {
                "source_url": item.get('source_url', ''),
                "lecture": item.get('lecture_id', '')
            }
        }
        test_dataset.append(entry)
        print(f"Text QA: {qa['question'][:50]}...")

print("Generating Hybrid Questions")
for item in samples_img:
    prec_text_id = item.get('preceding_text_id')
    prec_text = text_lookup.get(prec_text_id, "")

    # Эмулируем случайный выбор режима (как в проде)
    mode = "image" if random.random() < 0.3 else "hybrid"

    qa = generate_qa_pair(item, mode=mode, preceding_text=prec_text)
    if qa:
        # Собираем ID
        gold_ids = [item['id']]
        if mode == "hybrid" and prec_text_id:
            gold_ids.append(prec_text_id) # Текст тоже правильный ответ

        entry = {
            "question": qa['question'],
            "answer": qa['answer'],
            "type": mode,
            "gold_ids": gold_ids,
            "metadata": {
                "image_path": item['path'],
                "caption": item.get('caption', '')
            }
        }
        test_dataset.append(entry)
        print(f"{mode.upper()} QA: {qa['question'][:50]}")

print(f"Успешно сгенерировано: {len(test_dataset)}/10")
print("Пример финального JSON (проверьте структуру):")
print(json.dumps(test_dataset[-1], indent=2, ensure_ascii=False))

Generating Text Questions
Text QA: Какая единица измерения используется для измерения...
Text QA: Какие параметры характеризуют нелинейный резистивн...
Text QA: Как определить принужденную составляющую тока в це...
Text QA: Какое свойство ферромагнитного сердечника использу...
Generating Hybrid Questions
HYBRID QA: Какую емкость конденсатора С нужно выбрать, чтобы 
HYBRID QA: Каковы последствия отсутствия обратных волн тока и
Успешно сгенерировано: 6/10
Пример финального JSON (проверьте структуру):
{
  "question": "Каковы последствия отсутствия обратных волн тока и напряжения в решении уравнений линии бесконечной длины, и как это влияет на формулы (5) и (6) для напряжения и тока?",
  "answer": "Отсутствие обратных волн тока и напряжения в решении уравнений линии бесконечной длины приводит к тому, что слагаемые, содержащие экспоненту с отрицательным значением, должны быть исключены из выражений (5) и (6). Это связано с тем, что стремление этих слагаемых к нулю лишает их физического смыс

In [5]:
test_dataset

[{'question': 'Какая единица измерения используется для измерения мощности в трехфазных цепях?',
  'answer': 'Ватт',
  'type': 'text',
  'gold_ids': ['L18_p55'],
  'metadata': {'source_url': 'https://www.toehelp.ru/theory/toe/lecture18/lecture18.html',
   'lecture': 'L18'}},
 {'question': 'Какие параметры характеризуют нелинейный резистивный элемент?',
  'answer': 'Нелинейный резистивный элемент характеризуется статическим и дифференциальным сопротивлением.',
  'type': 'text',
  'gold_ids': ['L01_p17'],
  'metadata': {'source_url': 'https://www.toehelp.ru/theory/toe/lecture01/lecture01.html',
   'lecture': 'L01'}},
 {'question': 'Как определить принужденную составляющую тока в цепи?',
  'answer': 'Принужденная составляющая определяется соотношением',
  'type': 'text',
  'gold_ids': ['L38_p27'],
  'metadata': {'source_url': 'https://www.toehelp.ru/theory/toe/lecture38/lecture38.html',
   'lecture': 'L38'}},
 {'question': 'Какое свойство ферромагнитного сердечника используется в катушке 

Финальный запуск и генерация датасета для оцеки

In [ ]:
from tqdm.notebook import tqdm
import random
import json
import time

TARGET_COUNT = 1000
BATCH_SAVE = 10
OUTPUT_FILE = os.path.join(DATA_DIR, "benchmark_final.json")

# Загрузка
with open(os.path.join(DATA_DIR, "theory.json"), 'r') as f:
    theory_data = json.load(f)
with open(os.path.join(DATA_DIR, "images.json"), 'r') as f:
    images_data = json.load(f)

text_lookup = {t['id']: t['text'] for t in theory_data}
valid_images = [img for img in images_data if os.path.exists(img['path'])]

# Перемешиваем
random.shuffle(theory_data)
random.shuffle(valid_images)

gold_dataset = []
cnt_text = 0
cnt_visual = 0

print(f"Начинаем СМЕШАННУЮ генерацию {TARGET_COUNT} примеров...")

for i in range(TARGET_COUNT):
    print(f"{i+1}/{TARGET_COUNT} (Txt:{cnt_text} Img:{cnt_visual}) ...", end="\r")

    try:
        entry = None
        target_text = TARGET_COUNT // 2

        # Определяем, что генерить сейчас (бросаем монетку, но следим за лимитами)
        want_text = (random.random() < 0.5)

        # Если выпал текст И лимит не исчерпан ИЛИ картинки кончились -> делаем Текст
        if (want_text and cnt_text < target_text) or (cnt_visual >= len(valid_images)):
            if cnt_text < len(theory_data):
                item = theory_data[cnt_text]
                cnt_text += 1
                qa = generate_qa_pair(item, mode="text")
                if qa:
                    entry = {
                        "question": qa['question'], "answer": qa['answer'],
                        "type": "text", "gold_ids": [item['id']],
                        "metadata": {"source_id": item['id']}
                    }

        # Иначе -> делаем Картинку
        else:
            if cnt_visual < len(valid_images):
                item = valid_images[cnt_visual]
                cnt_visual += 1

                prec_text_id = item.get('preceding_text_id')
                prec_text = text_lookup.get(prec_text_id, "")
                mode = "image" if random.random() < 0.3 else "hybrid"

                qa = generate_qa_pair(item, mode=mode, preceding_text=prec_text)
                if qa:
                    gold_ids = [item['id']]
                    if mode == "hybrid" and prec_text_id: gold_ids.append(prec_text_id)

                    entry = {
                        "question": qa['question'], "answer": qa['answer'],
                        "type": mode, "gold_ids": gold_ids,
                        "metadata": {"image_path": item['path']}
                    }

        # Обработка результата
        if entry:
            gold_dataset.append(entry)
            # Выводим тип вопроса, чтобы видеть баланс
            print(f"[{len(gold_dataset)}] {entry['type'].upper()}: {entry['question'][:40]}...")
        else:
            pass

    except Exception as e:
        print(f"Error: {e}")
        continue

    # Сохранение
    if len(gold_dataset) % BATCH_SAVE == 0:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(gold_dataset, f, ensure_ascii=False, indent=4)

# Финал
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(gold_dataset, f, ensure_ascii=False, indent=4)

print(f"\nГОТОВО! Собрано: {len(gold_dataset)}")

Начинаем СМЕШАННУЮ генерацию 1000 примеров...
[1] TEXT: Какое значение времени t1 можно определи...
[2] TEXT: Какой знак используется для обозначения ...
[3] IMAGE: Какие элементы связаны между собой в это...
[4] HYBRID: Какой метод используется для расчета воз...
[5] TEXT: Какие уравнения описывают работу воздушн...
[6] TEXT: Какое внутреннее сопротивление имеет иде...
[7] TEXT: Какой из следующих терминов относится к ...
[8] TEXT: Какая ветвь изображена на рисунке 1?...
[9] TEXT: Как определить суммарную реактивную мощн...
[10] HYBRID: Какой угол сдвига между током и напряжен...
[11] IMAGE: Какие элементы соединены в этой схеме?...
[12] HYBRID: Какие параметры влияют на потери в стали...
[13] HYBRID: Каково напряжение на стыке линий в момен...
[14] HYBRID: Какой метод расчета используется для опр...
[15] HYBRID: Какое название получает режим работы схе...
[16] IMAGE: Какое значение имеет сопротивление R6H?...
[17] HYBRID: Каково значение к-й гармоники ЭДС в элек...
[18] IMAGE: Какие 